<a href="https://colab.research.google.com/github/ruudtje21/html-portofolio/blob/main/Sentymen_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Kode Python untuk Analisis Sentimen Tweet Maskapai Penerbangan


In [ ]:
!pip install langchain_community
!pip install replicate

In [32]:
from langchain_community.llms import Replicate
import os
from google.colab import userdata
# Set the API token
api_token = userdata.get('api_token')
os.environ["REPLICATE_API_TOKEN"] = api_token
# Model setup
model = "ibm-granite/granite-3.3-8b-instruct"
output = Replicate(
model=model,
replicate_api_token=api_token,
)

In [33]:
import pandas as pd
from collections import Counter
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

## Eksplorasi Data Awal

In [34]:
df = pd.read_csv("Tweets.csv")

In [35]:

print("Info Dataframe:")
print(df.info())

print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names:")
print(df.columns)

print("\nShape of the dataframe:")
print(df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nUnique values in relevant columns (e.g., airline_sentiment):")
print(df["airline_sentiment"].value_counts())

Info Dataframe:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   tweet_id                      14640 non-null  int64  
 1   airline_sentiment             14640 non-null  object 
 2   airline_sentiment_confidence  14640 non-null  float64
 3   negativereason                9178 non-null   object 
 4   negativereason_confidence     10522 non-null  float64
 5   airline                       14640 non-null  object 
 6   airline_sentiment_gold        40 non-null     object 
 7   name                          14640 non-null  object 
 8   negativereason_gold           32 non-null     object 
 9   retweet_count                 14640 non-null  int64  
 10  text                          14640 non-null  object 
 11  tweet_coord                   1019 non-null   object 
 12  tweet_created                 14640 non-null

## Pra-pemrosesan Data

In [36]:
# Download necessary NLTK data
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")
try:
    nltk.data.find("corpora/wordnet")
except LookupError:
    nltk.download("wordnet")

df = pd.read_csv("Tweets.csv")

# Drop columns with too many missing values or irrelevant for sentiment analysis
df = df.drop(columns=["negativereason_gold", "airline_sentiment_gold", "tweet_coord", "user_timezone", "tweet_location"], errors="ignore")

# Fill missing negativereason with 'No negative reason' for consistency
df["negativereason"] = df["negativereason"].fillna("No negative reason")
df["negativereason_confidence"] = df["negativereason_confidence"].fillna(0)

# Text cleaning function
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower() # Lowercase
    text = re.sub(r"@[A-Za-z0-9_]+", "", text) # Remove mentions
    text = re.sub(r"#", "", text) # Remove hashtag symbol
    text = re.sub(r"RT", "", text) # Remove RT
    text = re.sub(r"https?://\\S+|www\\.\\S+", "", text) # Remove URLs
    text = re.sub(r"[^a-z\\s]", "", text) # Remove non-alphabetic characters
    text = text.split() # Tokenize
    text = [lemmatizer.lemmatize(word) for word in text if word not in stop_words] # Remove stopwords and lemmatize
    text = " ".join(text) # Join back
    return text

df["cleaned_text"] = df["text"].apply(clean_text)

# Save preprocessed data
df.to_csv("Tweets_preprocessed.csv", index=False)

print("Preprocessing complete. Saved to /home/ubuntu/data/Tweets_preprocessed.csv")
print(df.head())
print(df.isnull().sum())

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Preprocessing complete. Saved to /home/ubuntu/data/Tweets_preprocessed.csv
             tweet_id airline_sentiment  airline_sentiment_confidence  \
0  570306133677760513           neutral                        1.0000   
1  570301130888122368          positive                        0.3486   
2  570301083672813571           neutral                        0.6837   
3  570301031407624196          negative                        1.0000   
4  570300817074462722          negative                        1.0000   

       negativereason  negativereason_confidence         airline        name  \
0  No negative reason                     0.0000  Virgin America     cairdin   
1  No negative reason                     0.0000  Virgin America    jnardino   
2  No negative reason                     0.0000  Virgin America  yvonnalynn   
3          Bad Flight                     0.7033  Virgin America    jnardino   
4          Can't Tell                     1.0000  Virgin America    jnardino   

   re

## Klasifikasi Sentimen

In [37]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
import joblib

In [38]:
df = pd.read_csv("Tweets_preprocessed.csv")

X = df["cleaned_text"]
y = df["airline_sentiment"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train.fillna(''))
X_test_vec = vectorizer.transform(X_test.fillna(''))

model = MultinomialNB()
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Save model and vectorizer to the current working directory
joblib.dump(model, "sentiment_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("Model and vectorizer saved.")

Accuracy: 0.6547131147540983

Classification Report:
              precision    recall  f1-score   support

    negative       0.65      1.00      0.79      1889
     neutral       1.00      0.01      0.02       580
    positive       1.00      0.05      0.09       459

    accuracy                           0.65      2928
   macro avg       0.88      0.35      0.30      2928
weighted avg       0.78      0.65      0.53      2928

Model and vectorizer saved.


## Mengekstraksi Wawasan dan Ringkasan

In [39]:
# Download necessary NLTK data
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")
try:
    nltk.data.find("corpora/wordnet")
except LookupError:
    nltk.download("wordnet")

df = pd.read_csv("Tweets_preprocessed.csv")

# --- Insight Extraction: Most common words for each sentiment ---

def get_top_n_words(corpus, n=None):
    words = [word for sentence in corpus for word in sentence.split()]
    return Counter(words).most_common(n)

positive_tweets = df[df["airline_sentiment"] == "positive"]["cleaned_text"].dropna()
neutral_tweets = df[df["airline_sentiment"] == "neutral"]["cleaned_text"].dropna()
negative_tweets = df[df["airline_sentiment"] == "negative"]["cleaned_text"].dropna()

print("\n--- Top 20 words in Positive Tweets ---")
print(get_top_n_words(positive_tweets, 20))

print("\n--- Top 20 words in Neutral Tweets ---")
print(get_top_n_words(neutral_tweets, 20))

print("\n--- Top 20 words in Negative Tweets ---")
print(get_top_n_words(negative_tweets, 20))

print("\n--- Summarization of Sample Negative Tweets (Simple Approach) ---")

sample_negative_tweets_for_summary = df[df["airline_sentiment"] == "negative"]["text"].sample(min(5, len(negative_tweets)), random_state=42).tolist()

for i, tweet in enumerate(sample_negative_tweets_for_summary):
    print(f"Summary Sentence {i+1}: {tweet}")


--- Top 20 words in Positive Tweets ---
[('thankyou', 55), ('thanks', 52), ('thankssomuch', 7), ('greatthankyou', 5), ('awesomethanks', 4), ('hijustwantedtoseeifyouhaveanynewroutesplannedthisyearfornewarkloveflyingyouguysandhopetodosomore', 3), ('youremyearlyfrontrunnerforbestairlineoscars', 3), ('thnx', 3), ('iwillthankyou', 3), ('yesplease', 3), ('okthankyou', 3), ('greatthanks', 3), ('youarethebestfollowmeplease', 3), ('thanksfortheupdate', 2), ('thanksimadeit', 2), ('yourterryisourherogotmyhusbandbackthrusecuritytoretrievecellphoneleftonplaneinaustinterryurock', 2), ('signmeup', 2), ('madlovehttptcoojrsdwpkknyc', 2), ('awesomethankyou', 2), ('thankyouverymuch', 2)]

--- Top 20 words in Neutral Tweets ---
[('sent', 10), ('thankyou', 9), ('done', 9), ('didyouknowthatsuicideisthesecondleadingcauseofdeathamongteens', 6), ('flight', 6), ('dmsent', 4), ('higuysdoyouhaveageneralenquiresemailaddresspleasethanksdavid', 4), ('nothanks', 4), ('thanks', 4), ('iknowthisisprobablyanobutistherea

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Visualisasi Data

In [40]:
import matplotlib.pyplot as plt
import seaborn as sns

In [41]:
# Load preprocessed data
df = pd.read_csv("Tweets_preprocessed.csv")

# Set style for plots
sns.set_style("whitegrid")

# 1. Sentiment Distribution
plt.figure(figsize=(8, 6))
sns.countplot(x="airline_sentiment", data=df, palette="viridis")
plt.title("Distribusi Sentimen Maskapai Penerbangan")
plt.xlabel("Sentimen")
plt.ylabel("Jumlah Tweet")
plt.savefig("sentiment_distribution.png")
plt.close()

print("Sentiment distribution plot saved to /home/ubuntu/sentiment_distribution.png")

# 2. Top Negative Reasons
negative_reasons = df[df["airline_sentiment"] == "negative"]["negativereason"].value_counts().head(10)

plt.figure(figsize=(10, 7))
sns.barplot(x=negative_reasons.values, y=negative_reasons.index, palette="magma")
plt.title("Top 10 Alasan Negatif")
plt.xlabel("Jumlah Tweet")
plt.ylabel("Alasan Negatif")
plt.tight_layout()
plt.savefig("top_negative_reasons.png")
plt.close()

print("Top negative reasons plot saved to /home/ubuntu/top_negative_reasons.png")

/tmp/ipython-input-41-3266267248.py:9: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(x="airline_sentiment", data=df, palette="viridis")
/tmp/ipython-input-41-3266267248.py:22: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=negative_reasons.values, y=negative_reasons.index, palette="magma")


Sentiment distribution plot saved to /home/ubuntu/sentiment_distribution.png
Top negative reasons plot saved to /home/ubuntu/top_negative_reasons.png


## Menentukan contoh daftar ulasan pelanggan dan menyempurnakan permintaan untuk menyertakan ulasan

In [44]:
# Download necessary NLTK data
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")
try:
    nltk.data.find("corpora/wordnet")
except LookupError:
    nltk.download("wordnet")

# Define a sample list of customer reviews
customer_reviews = [
    "This flight was amazing, great service!",
    "The flight was delayed for hours, very frustrating.",
    "The staff was okay, nothing special.",
    "Lost my luggage, terrible experience.",
    "Smooth flight and friendly crew."
]

# Refine the prompt to include reviews
reviews_text = "\n".join([f"Review {i+1}: {review}" for i, review
in enumerate(customer_reviews)])
prompt = f"""
Classify these reviews as Positive, Negative, or Mixed:
{reviews_text}
"""
# Invoke the model with the example prompt
response = output.invoke(prompt)
# Print the response
print("Granite Model Response:\n")
print(response)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Granite Model Response:

1. Positive
2. Negative
3. Mixed (slightly leaning towards Negative due to the "nothing special" comment)
4. Negative
5. Positive


## Menentukan perintah untuk menyelesaikan tugas dalam 2 langkah

In [51]:
# Define the prompt to complete the task in 2 steps
multitask_prompt = f"""
Complete the task in 2 steps.
Step 1: Classify these reviews as positive, negative, or mixed.
Step 2: For each review, identify relevant categories: airline_sentiment, negativereason, tweet content
{reviews_text}
"""
response = output.invoke(multitask_prompt)
print("Granite Model Response:\n")
print(response)

Granite Model Response:

**Step 1: Classification**

1. Review 1: Positive
2. Review 2: Negative
3. Review 3: Mixed
4. Review 4: Negative
5. Review 5: Positive

**Step 2: Identification of Relevant Categories**

1. **Review 1: Positive**
   - airline_sentiment: Positive
   - negativereason: None
   - tweet_content: "This flight was amazing, great service!"

2. **Review 2: Negative**
   - airline_sentiment: Negative
   - negativereason: Flight delay
   - tweet_content: "The flight was delayed for hours, very frustrating."

3. **Review 3: Mixed**
   - airline_sentiment: Mixed
   - negativereason: Staff not exceptional
   - tweet_content: "The staff was okay, nothing special."

4. **Review 4: Negative**
   - airline_sentiment: Negative
   - negativereason: Lost luggage
   - tweet_content: "Lost my luggage, terrible experience."

5. **Review 5: Positive**
   - airline_sentiment: Positive
   - negativereason: None
   - tweet_content: "Smooth flight and friendly crew."
